# HCC behavioral analysis
Run the notebook from top to bottom. Edit only the settings cell for routine use.

**Views:** selected run; selected subject with all included runs pooled; all subjects in one panel grid; group mean; run-by-run trends; paired condition differences.

- Run numbers are **1–12**.
- Exclude the first two trials of each run. Primary RT uses only trials answered correctly, with finite and positive response times, converted to milliseconds. Missing responses and incorrect trials are excluded from RT. Accuracy uses all remaining trials, including errors and missing responses.
- Subject bars pool correct-trial RTs across included runs. Group bars average subject means with equal subject weights; group error bars show between-subject SEM.
- Missing and excluded runs are never replaced with zeros. A subject with no included data is omitted from analysis.

Dependencies: `numpy`, `pandas`, `matplotlib`, and `IPython`. Use a kernel with access to the shared data folder. Restart the kernel and run all cells after changing settings.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Data and individual view.
DATA_ROOT = Path('/data/backed_up/shared/ME_7T_Pilot/behav_data')
SELECTED_SUBJECT = '11113'
SELECTED_RUN = 1  # Actual run number: 1–12.
SUBJECTS = None  # None includes all subjects; otherwise use a list of string IDs.

# Exclusions apply to every table and plot. Example: {'11084': [12]}.
EXCLUDE_RUNS = {'11113':[1,2]}
N_INITIAL_STAYS = 2
REQUIRE_ALL_12_RUNS = False  # Applies only to the group mean.

# Resolve duplicate files with (subject, actual run): exact filename.
FILE_OVERRIDES = {}

# Optional exports. Existing files with the same names will be overwritten.
SAVE_RESULTS = False
OUTPUT_DIR = Path.cwd() / 'hcc_results'
PANEL_COLUMNS = 3

RUNS = list(range(1, 13))
TRIAL_TYPES = ['EDS', 'IDS', 'Stay']
COLORS = ['#4878D0', '#EE854A', '#6ACC64']
METRICS = [('mean_RT_ms', 'Reaction time (ms)'), ('accuracy_pct', 'Accuracy (%)')]
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})


## 1. Data helpers
These functions handle discovery, exclusions, validation, and summaries. All file searches are recursive. Duplicate subject/run files require an explicit selection.


In [ ]:
FILE_PATTERN = re.compile(
    r'^sub-(?P<subject>.+)_task-HCC_block-(?P<block>\d+)_date-(?P<date>\d{8}_\d{4})\.csv$'
)


def discover_files():
    """Find one included CSV for each subject and run."""
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f'The current kernel cannot access {DATA_ROOT}')
    if SELECTED_RUN not in RUNS:
        raise ValueError('SELECTED_RUN must be between 1 and 12.')
    for subject, runs in EXCLUDE_RUNS.items():
        if not isinstance(subject, str) or any(type(run) is not int or run not in RUNS for run in runs):
            raise ValueError('EXCLUDE_RUNS requires string IDs and integer runs from 1 to 12.')

    records = []
    for path in sorted(DATA_ROOT.rglob('sub-*_task-HCC_block-*_date-*.csv')):
        match = FILE_PATTERN.fullmatch(path.name)
        if match is None:
            continue
        subject, run = match['subject'], int(match['block']) + 1
        if run in RUNS and (SUBJECTS is None or subject in SUBJECTS):
            records.append(dict(subject=subject, run=run, date=match['date'], path=path))

    found = pd.DataFrame(records, columns=['subject', 'run', 'date', 'path'])
    subjects = sorted(set(found['subject']) | set(SUBJECTS or []))
    coverage_rows, included_rows, excluded_rows = [], [], []
    for subject in subjects:
        files = found.loc[found['subject'] == subject]
        present = set(files['run'])
        excluded = set(EXCLUDE_RUNS.get(subject, []))
        coverage_rows.append({
            'subject': subject,
            'n_runs': len(present - excluded),
            'included_runs': sorted(present - excluded),
            'missing_runs': sorted(set(RUNS) - present),
            'excluded_runs': sorted(excluded),
            'excluded_found_runs': sorted(present & excluded),
        })
        for run, candidates in files.groupby('run'):
            if run in excluded:
                excluded_rows.extend(candidates.to_dict('records'))
                continue
            override = FILE_OVERRIDES.get((subject, run))
            if override is not None:
                candidates = candidates.loc[candidates['path'].map(lambda path: path.name == override)]
            if len(candidates) != 1:
                names = [str(path) for path in candidates['path']]
                raise ValueError(f'{subject}, run {run}: expected one file; found {names}. Check FILE_OVERRIDES.')
            included_rows.extend(candidates.to_dict('records'))

    for (subject, run), filename in FILE_OVERRIDES.items():
        if SUBJECTS is not None and subject not in SUBJECTS:
            continue
        if run not in EXCLUDE_RUNS.get(subject, []) and not any(
            row['subject'] == subject and row['run'] == run for row in included_rows
        ):
            raise ValueError(f'Override not found: {subject}, run {run}, {filename}')
    unknown = set(EXCLUDE_RUNS) - set(subjects)
    if unknown:
        warnings.warn(f'Exclusion settings refer to unavailable subjects: {sorted(unknown)}')
    coverage = pd.DataFrame(coverage_rows)
    if not included_rows:
        display(coverage)
        raise ValueError('No included files. Check the data folder and selection settings.')
    manifest = pd.DataFrame(included_rows).sort_values(['subject', 'run']).reset_index(drop=True)
    return manifest, pd.DataFrame(excluded_rows, columns=found.columns), coverage


def load_trials(manifest):
    """Load trials and calculate accuracy and correct-trial RT."""
    frames = []
    required = {'trial', 'hierarhical_switch_type', 'subject_correct', 'response_time'}
    for row in manifest.itertuples(index=False):
        frame = pd.read_csv(row.path)
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f'{row.path.name}: missing columns {missing}')
        for column in ['trial', 'subject_correct', 'response_time']:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
        if frame['trial'].duplicated().any() or not frame['trial'].isin(range(1, 39)).all():
            raise ValueError(f'{row.path.name}: trial IDs must be unique integers from 1 to 38.')
        if len(frame) != 38:
            warnings.warn(f'{row.path.name}: found {len(frame)} trials; expected 38.')
        if not frame['subject_correct'].dropna().isin([0, 1]).all():
            raise ValueError(f'{row.path.name}: subject_correct must contain 0, 1, or missing values.')
        frame['trial_type'] = frame['hierarhical_switch_type'].astype('string').str.strip()
        if not frame['trial_type'].isin(TRIAL_TYPES).all():
            raise ValueError(f'{row.path.name}: unknown trial types.')
        frames.append(frame.assign(subject=row.subject, run=row.run, source_file=row.path.name))

    data = pd.concat(frames, ignore_index=True)
    data = data.loc[data['trial'] > N_INITIAL_STAYS].copy()
    data['correct'] = data['subject_correct'].eq(1)
    valid_rt = np.isfinite(data['response_time']) & data['response_time'].gt(0)
    # RT is based only on correct trials; accuracy keeps all retained trials.
    data['RT_ms'] = (1000 * data['response_time']).where(valid_rt & data['correct'])
    return data


def summarize(data, keys):
    """Pool trials by subject/run and trial type."""
    result = data.groupby(keys + ['trial_type'], observed=True).agg(
        n_trials=('correct', 'size'), n_correct=('correct', 'sum'),
        n_rt=('RT_ms', 'count'), mean_RT_ms=('RT_ms', 'mean'),
        median_RT_ms=('RT_ms', 'median'), SD_RT_ms=('RT_ms', 'std'),
    ).reset_index()
    # Accuracy uses all retained trials; RT uses valid correct trials only.
    result['accuracy_pct'] = 100 * result['n_correct'] / result['n_trials']
    return result

## 2. Load data and check coverage
`included_runs` lists actual run numbers used in analysis. `missing_runs` and `excluded_runs` are reported separately. The tables use `n_trials` as the accuracy denominator and `n_rt` as the RT sample size.


In [ ]:
manifest, excluded_files, coverage = discover_files()
data = load_trials(manifest)
run_summary = summarize(data, ['subject', 'run'])
subject_summary = summarize(data, ['subject'])

eligible = coverage.loc[coverage['n_runs'] == 12, 'subject'] if REQUIRE_ALL_12_RUNS else coverage['subject']
group_subject_summary = subject_summary.loc[subject_summary['subject'].isin(eligible)].copy()
# Group means give every subject equal weight; SEM is the between-subject standard error.
group_summary = group_subject_summary.groupby('trial_type').agg(
    n_subjects_RT=('mean_RT_ms', 'count'), mean_RT_ms=('mean_RT_ms', 'mean'),
    SEM_RT_ms=('mean_RT_ms', 'sem'),
    n_subjects_accuracy=('accuracy_pct', 'count'), accuracy_pct=('accuracy_pct', 'mean'),
    SEM_accuracy_pct=('accuracy_pct', 'sem'),
).reindex(TRIAL_TYPES)

print(f'Loaded {len(manifest)} runs from {manifest.subject.nunique()} subjects.')
display(coverage)
if not excluded_files.empty:
    display(excluded_files)
figures = {}

## 3. Plot helpers
Bars use the same condition colors throughout. Subject panels share a scale within each metric. Missing values are labeled “No data”; they are not zero-valued bars.


In [ ]:
def draw_bars(ax, table, metric, title, points=None, sem=None, run_points=None):
    """Draw condition bars and optional subject/run observations."""
    table = table.reindex(TRIAL_TYPES)
    x = np.arange(len(TRIAL_TYPES))
    values = table[metric].to_numpy(dtype=float)
    ax.bar(x, values, color=COLORS, width=0.65, alpha=0.85)

    if sem is not None:
        errors = table[sem].to_numpy(dtype=float)
        valid = np.isfinite(values) & np.isfinite(errors)
        ax.errorbar(x[valid], values[valid], yerr=errors[valid], fmt='none', color='black', capsize=4)
    if points is not None:
        for i, condition in enumerate(TRIAL_TYPES):
            ys = points.loc[points['trial_type'] == condition, metric].dropna().to_numpy()
            offsets = np.linspace(-0.16, 0.16, len(ys)) if len(ys) > 1 else np.zeros(len(ys))
            ax.scatter(i + offsets, ys, s=22, color='black', alpha=0.55, zorder=3)
    if run_points is not None and not run_points.empty:
        run_wide = run_points.pivot(index='run', columns='trial_type', values=metric).reindex(columns=TRIAL_TYPES)
        for run_values in run_wide.to_numpy(dtype=float):
            valid = np.isfinite(run_values)
            if valid.any():
                ax.plot(x[valid], run_values[valid], color='black', marker='o',
                        markersize=4, linewidth=1, alpha=0.4, zorder=4)

    for i, value in enumerate(values):
        if np.isfinite(value):
            ax.annotate(f'{value:.1f}', (i, value), xytext=(0, 6),
                        textcoords='offset points', ha='center', fontsize=10)
        else:
            ax.text(i, 0.02, 'No data', transform=ax.get_xaxis_transform(), ha='center', fontsize=9)
    ax.set_xticks(x, TRIAL_TYPES)
    ax.set_title(title)
    ax.set_ylabel(dict(METRICS)[metric])
    ax.grid(axis='y', alpha=0.15)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)
    if metric == 'accuracy_pct':
        ax.set_ylim(0, 112)
        ax.set_yticks(range(0, 101, 20))
    else:
        ax.margins(y=0.22)


def plot_pair(table, title, points=None, group=False, run_points=None):
    """Show RT and accuracy side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout='constrained')
    for ax, (metric, label) in zip(axes, METRICS):
        sem = {'mean_RT_ms': 'SEM_RT_ms', 'accuracy_pct': 'SEM_accuracy_pct'}[metric] if group else None
        draw_bars(ax, table, metric, label, points=points, sem=sem, run_points=run_points)
    fig.suptitle(title, fontsize=14)
    return fig


def plot_subject_panels(summary, metric):
    """Show pooled bars for every subject."""
    subjects = sorted(summary['subject'].unique())
    if not subjects:
        return None
    ncols = max(1, min(PANEL_COLUMNS, len(subjects)))
    nrows = int(np.ceil(len(subjects) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows),
                             sharey=True, squeeze=False, layout='constrained')
    run_counts = coverage.set_index('subject')['n_runs']
    rt_top = max(1.0, summary[metric].max() * 1.25)
    for ax, subject in zip(axes.flat, subjects):
        table = summary.loc[summary['subject'] == subject].set_index('trial_type')
        subject_runs = run_summary.loc[run_summary['subject'] == subject]
        draw_bars(ax, table, metric, f'{subject} | {run_counts[subject]}/12 runs', run_points=subject_runs)
        if metric == 'mean_RT_ms':
            ax.set_ylim(0, rt_top)
    for ax in list(axes.flat)[len(subjects):]:
        ax.set_visible(False)
    fig.suptitle(f'All subjects | {dict(METRICS)[metric]}', fontsize=14)
    return fig

## 4. Selected subject: all included runs pooled
Each bar pools the selected subject's trials across all included runs. RT and accuracy are shown side by side.


In [ ]:
selected = subject_summary.loc[subject_summary['subject'] == SELECTED_SUBJECT]
selected_runs = run_summary.loc[run_summary['subject'] == SELECTED_SUBJECT]
if selected.empty:
    print(f'No included data for subject {SELECTED_SUBJECT}.')
else:
    n_runs = int(coverage.set_index('subject').loc[SELECTED_SUBJECT, 'n_runs'])
    display(selected.round(2))
    figures['selected_subject_pooled'] = plot_pair(
        selected.set_index('trial_type'),
        f'{SELECTED_SUBJECT} | All included runs pooled ({n_runs}/12) | Run points and lines',
        run_points=selected_runs,
    )
    plt.show()


## 5. All subjects at a glance
Every subject with included data gets a separate bar chart, arranged in a single grid per metric. The first figure shows RT; the second shows accuracy. This view includes subjects with incomplete runs even when the group mean requires all 12 runs.


In [ ]:
for metric, label in METRICS:
    figure = plot_subject_panels(subject_summary, metric)
    if figure is not None:
        figures[f'all_subjects_{metric}'] = figure
        plt.show()


## 6. Group mean
Each subject has equal weight. Black dots show subject means and error bars show between-subject SEM. SEM is undefined when fewer than two subjects contribute to a condition. The table reports separate subject counts for RT and accuracy.


In [ ]:
if group_subject_summary.empty:
    print('No subjects meet the group inclusion settings.')
else:
    display(group_summary.round(2))
    n_subjects = group_subject_summary.subject.nunique()
    figures['group_mean'] = plot_pair(
        group_summary, f'Group mean | {n_subjects} subjects',
        points=group_subject_summary, group=True,
    )
    plt.show()


## 7. Paired condition differences
For each subject, calculate EDS - IDS and EDS - Stay differences. Positive RT values mean EDS is slower; positive accuracy values mean EDS is more accurate. Points show individual subjects, bars show the mean difference, and the horizontal line marks zero.

In [ ]:
# Positive RT difference means EDS was slower; positive accuracy difference means EDS was more accurate.
paired = group_subject_summary.pivot(
    index='subject', columns='trial_type', values=['mean_RT_ms', 'accuracy_pct']
)
contrasts = {'EDS - IDS': ('EDS', 'IDS'), 'EDS - Stay': ('EDS', 'Stay')}
differences = pd.DataFrame(index=paired.index)
for metric, _ in METRICS:
    for name, (left, right) in contrasts.items():
        differences[f'{metric} | {name}'] = paired[(metric, left)] - paired[(metric, right)]

rows = []
for metric, label in METRICS:
    for name in contrasts:
        values = differences[f'{metric} | {name}'].dropna()
        rows.append({
            'metric': label,
            'contrast': name,
            'n_subjects': len(values),
            'mean_difference': values.mean(),
            'SEM': values.sem(),
            'median_difference': values.median(),
            'n_positive': int((values > 0).sum()),
            'n_negative': int((values < 0).sum()),
        })
contrast_summary = pd.DataFrame(rows)
display(contrast_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
x = np.arange(len(contrasts))
for ax, (metric, label) in zip(axes, METRICS):
    means, errors = [], []
    for i, name in enumerate(contrasts):
        values = differences[f'{metric} | {name}'].dropna()
        means.append(values.mean())
        errors.append(values.sem())
        jitter = np.linspace(-0.12, 0.12, len(values)) if len(values) > 1 else np.zeros(len(values))
        ax.scatter(i + jitter, values, color='black', s=28, alpha=0.7, zorder=3)
    ax.bar(x, means, yerr=errors, color=['#4878D0', '#6ACC64'], alpha=0.8, capsize=4)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x, contrasts)
    ax.set_ylabel(f'{label} difference')
    ax.set_title(label)
    ax.grid(axis='y', alpha=0.15)
    ax.set_axisbelow(True)
fig.suptitle(f'Paired condition differences | {len(differences)} subjects')
figures['paired_condition_differences'] = fig
plt.show()

## 8. Selected run
Use `SELECTED_SUBJECT` and `SELECTED_RUN` in the settings cell to select an individual run.

In [ ]:
individual = run_summary.loc[
    (run_summary['subject'] == SELECTED_SUBJECT) & (run_summary['run'] == SELECTED_RUN)
]
if individual.empty:
    print(f'No included data for {SELECTED_SUBJECT}, run {SELECTED_RUN}.')
else:
    display(individual.round(2))
    figures['selected_run'] = plot_pair(
        individual.set_index('trial_type'), f'{SELECTED_SUBJECT} | Run {SELECTED_RUN}'
    )
    plt.show()


## 9. Run-by-run trends
Missing or excluded runs appear as gaps. Run numbers always range from 1 to 12.

In [ ]:
selected_runs = run_summary.loc[run_summary['subject'] == SELECTED_SUBJECT]
if not selected_runs.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')
    for ax, (metric, label) in zip(axes, METRICS):
        wide = selected_runs.pivot(index='run', columns='trial_type', values=metric)
        wide = wide.reindex(index=RUNS, columns=TRIAL_TYPES)
        for condition, color in zip(TRIAL_TYPES, COLORS):
            ax.plot(wide.index, wide[condition], marker='o', color=color, label=condition)
        ax.set(xlabel='Run', ylabel=label, xticks=RUNS)
        ax.set_ylim(bottom=0)
        if metric == 'accuracy_pct':
            ax.set_ylim(0, 105)
        ax.grid(axis='y', alpha=0.15)
        ax.legend()
    fig.suptitle(f'{SELECTED_SUBJECT} | Run-by-run trends')
    figures['run_trends'] = fig
    plt.show()


## 10. Optional export
Set `SAVE_RESULTS = True` to save summary CSVs and all displayed figures as PNGs. Raw data are never modified. Existing exports with matching filenames are overwritten.

In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {
        'file_manifest': manifest, 'excluded_files': excluded_files,
        'run_coverage': coverage, 'run_summary': run_summary,
        'subject_summary': subject_summary, 'group_included_subjects': group_subject_summary,
        'group_summary': group_summary.reset_index(),
    }
    for name, table in tables.items():
        table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
    for name, figure in figures.items():
        figure.savefig(OUTPUT_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
    print(f'Saved tables and figures to {OUTPUT_DIR.resolve()}')
